In [0]:
%run ../helpers/common_utilities

In [0]:
volume_location = f"/Volumes/{CATALOG_NAME}/{Default_schema}/{Volume_name}"
print(volume_location)

In [0]:
query = """select sensor_event_id,
            alarm_count,
            cycle_count,
            alarm_count * 0.1 AS estimated_defects,

                CASE
                    WHEN cycle_count > 0 THEN
                        (
                            cycle_count - (alarm_count * 0.1)
                        ) / cycle_count
                    ELSE 0
                END AS quality_rate,
                publish_timestamp,
                current_timestamp() as Load_Timestamp
                from temp_view"""

Silver_Table_dict = {
    'CNC':{
        'source_table_name':"iot_sensor_catalog.bronze_iot.cnc_sensor_cdc",
        'checkpoint_location':f"{volume_location}/checkpoints/cnc_sensor_quality_metrics",
        'target_table':f"iot_sensor_catalog.silver_iot.T_cnc_quality_metrics",
        'query':query
    },

    'Motor':{
        'source_table_name':"iot_sensor_catalog.bronze_iot.motor_sensor_cdc",
        'checkpoint_location':f"{volume_location}/checkpoints/motor_sensor_quality_metrics",
        'target_table':f"iot_sensor_catalog.silver_iot.T_motor_quality_metrics",
        'query':query
    },

    'Transformer':{
        'source_table_name':"iot_sensor_catalog.bronze_iot.transformer_sensor_cdc",
        'checkpoint_location':f"{volume_location}/checkpoints/transformer_sensor_quality_metrics",
        'target_table':f"iot_sensor_catalog.silver_iot.T_transformer_quality_metrics",
        'query':query
    }
}
from concurrent.futures import ThreadPoolExecutor

def run_stream_processor(table_key):
    print(f'started for {table_key}')
    config = Silver_Table_dict[table_key]
    print(f'''started for {table_key}
          source_table_name={config['source_table_name']},
        checkpoint_location={config['checkpoint_location']},
        target_table={config['target_table']},
        query={config['query']}
          ''')
    processor = StreamProcessor(
        source_table_name=config['source_table_name'],
        checkpoint_location=config['checkpoint_location'],
        target_table=config['target_table'],
        query=config['query']
    )
    df = processor.read_stream()
    query = processor.write_stream(
        df=df,
        output_mode="append",
        trigger_type='availableNow'
    )
    query.awaitTermination()

with ThreadPoolExecutor() as executor:
    executor.map(run_stream_processor, Silver_Table_dict.keys())
